In [1]:
!wget -O banglaclip_model_epoch_10.pth https://huggingface.co/Mansuba/BanglaCLIP13/resolve/main/banglaclip_model_epoch_10.pth


--2025-01-21 10:52:14--  https://huggingface.co/Mansuba/BanglaCLIP13/resolve/main/banglaclip_model_epoch_10.pth
Resolving huggingface.co (huggingface.co)... 18.239.50.49, 18.239.50.103, 18.239.50.16, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.49|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/2a/ad/2aadee49f0f0c86b7b267a72f0937b06e9c98f694b60f9d09a1698caad62340d/f25c01d0773579e603903fefc52f721337cf92cbcbfb4ab6e48d2c858c8cbc3f?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27banglaclip_model_epoch_10.pth%3B+filename%3D%22banglaclip_model_epoch_10.pth%22%3B&Expires=1737460334&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTczNzQ2MDMzNH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzJhL2FkLzJhYWRlZTQ5ZjBmMGM4NmI3YjI2N2E3MmYwOTM3YjA2ZTljOThmNjk0YjYwZjlkMDlhMTY5OGNhYWQ2MjM0MGQvZjI1YzAxZDA3NzM1NzllNjAzOTAzZmVmYzUyZjcyMTMzN2NmOTJjYmNiZmI0YWI2ZT

#Result with banglaclip

In [7]:
import torch
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import numpy as np
from typing import List, Tuple, Optional

class EnhancedBanglaSDGenerator:
    def __init__(self, banglaclip_weights_path: str, device: Optional[torch.device] = None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Initialize models and processors
        self.clip_model_name = "openai/clip-vit-base-patch32"
        self.bangla_text_model = "csebuetnlp/banglabert"

        # Load BanglaCLIP with improved initialization
        self.banglaclip_model = self._load_banglaclip_model(banglaclip_weights_path)
        self.processor = CLIPProcessor.from_pretrained(self.clip_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(self.bangla_text_model)

        # Enhanced Stable Diffusion initialization
        self.pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            safety_checker=None  # Disable safety checker for better performance
        )
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            self.pipe.scheduler.config,
            use_karras_sigmas=True,  # Enable Karras sigmas for better quality
            algorithm_type="dpmsolver++"  # Use improved solver
        )
        self.pipe = self.pipe.to(self.device)

        # Move models to device
        self.banglaclip_model = self.banglaclip_model.to(self.device)

        # Initialize translation mapping
        self.translation_map = {
            'প্রাকৃতিক': 'natural',
            'সুন্দর': 'beautiful',
            'দৃশ্য': 'scene',
            'পাহাড়': 'mountain',
            'সূর্য': 'sun',
            'আকাশ': 'sky',
            'মেঘ': 'cloud',
            'নদী': 'river',
            'সমুদ্র': 'ocean',
            'গাছ': 'tree',
            'ফুল': 'flower'
            # Add more mappings as needed
        }

    def _load_banglaclip_model(self, weights_path: str) -> CLIPModel:
        """Enhanced model loading with better error handling"""
        try:
            clip_model = CLIPModel.from_pretrained(self.clip_model_name)
            state_dict = torch.load(weights_path, map_location=self.device)

            # Enhanced state dict cleaning
            cleaned_state_dict = {}
            for k, v in state_dict.items():
                k = k.replace('module.', '')
                k = k.replace('clip.', '')
                if k.startswith('text_model.') or k.startswith('vision_model.'):
                    cleaned_state_dict[k] = v

            clip_model.load_state_dict(cleaned_state_dict, strict=False)
            return clip_model
        except Exception as e:
            raise RuntimeError(f"Failed to load BanglaCLIP model: {str(e)}")

    def _get_text_embedding(self, bangla_text: str) -> torch.Tensor:
        """Get enhanced text embeddings with improved tokenization"""
        inputs = self.tokenizer(
            bangla_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77,
            return_attention_mask=True
        ).to(self.device)

        inputs.pop("token_type_ids", None)

        with torch.no_grad():
            text_features = self.banglaclip_model.get_text_features(**inputs)
            # Normalize features for better similarity matching
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        return text_features

    def _enhance_prompt_with_clip(self, text_features: torch.Tensor, original_text: str) -> str:
        """Enhanced prompt generation using CLIP features and context"""
        # Basic translation
        words = original_text.split()
        translated_parts = []
        for word in words:
            if word in self.translation_map:
                translated_parts.append(self.translation_map[word])
            else:
                translated_parts.append(word)

        base_translation = " ".join(translated_parts)

        # Content-based enhancements
        style_keywords = ["high quality", "detailed", "professional photography", "4k", "sharp focus"]

        # Add context-specific enhancements
        if any(word in original_text for word in ['প্রাকৃতিক', 'দৃশ্য']):
            style_keywords.extend([
                "landscape photography",
                "dramatic lighting",
                "golden hour",
                "cinematic",
                "high resolution"
            ])

        if any(word in original_text for word in ['সূর্য', 'আকাশ']):
            style_keywords.extend([
                "atmospheric",
                "beautiful sky",
                "natural lighting",
                "HDR",
                "vivid colors"
            ])

        # Combine everything into an enhanced prompt
        enhanced_prompt = f"{base_translation}, {', '.join(style_keywords)}"
        return enhanced_prompt

    def generate_image(
        self,
        bangla_text: str,
        num_images: int = 1,
        num_inference_steps: int = 50,
        guidance_scale: float = 8.5,
        seed: Optional[int] = None
    ) -> Tuple[List[any], str]:
        """Generate images with enhanced settings and controls"""
        try:
            # Set random seed if provided
            if seed is not None:
                torch.manual_seed(seed)

            # Get enhanced text embeddings
            text_features = self._get_text_embedding(bangla_text)

            # Generate enhanced prompt
            enhanced_prompt = self._enhance_prompt_with_clip(text_features, bangla_text)

            # Enhanced negative prompt
            negative_prompt = (
                "blurry, low quality, pixelated, cartoon, fuzzy, noisy, "
                "oversaturated, undersaturated, distorted, deformed, "
                "bad anatomy, watermark, signature, text"
            )

            # Generate image with enhanced settings
            with torch.autocast(self.device.type):
                result = self.pipe(
                    prompt=enhanced_prompt,
                    negative_prompt=negative_prompt,
                    num_images_per_prompt=num_images,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale
                )

            return result.images, enhanced_prompt

        except Exception as e:
            print(f"Error during image generation: {str(e)}")
            return None, None

# Example usage
if __name__ == "__main__":
    generator = EnhancedBanglaSDGenerator(
        banglaclip_weights_path="/content/banglaclip_model_epoch_10.pth"
    )

    # Example Bangla text
    bangla_text = "শিশির ভেজা ফুল"  # A beautiful natural scene

    # Generate multiple images with enhanced settings
    images, used_prompt = generator.generate_image(
        bangla_text,
        num_images=2,
        num_inference_steps=50,
        guidance_scale=8.5,
        seed=42  # Set seed for reproducibility
    )

    # Save generated images
    if images:
        for idx, image in enumerate(images):
            image.save(f"generated_image_banglaclip_{idx}.png")
            print(f"Image {idx} saved with prompt: {used_prompt}")
    else:
        print("Image generation failed")

<ipython-input-7-06480b57ffbc>:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weights_path, map_location=self.device)


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  0%|          | 0/50 [00:00<?, ?it/s]

Image 0 saved with prompt: শিশির ভেজা flower, high quality, detailed, professional photography, 4k, sharp focus
Image 1 saved with prompt: শিশির ভেজা flower, high quality, detailed, professional photography, 4k, sharp focus


#Result with banglabert

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel, AutoTokenizer
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
# Load Pretrained Models
clip_model_name = "openai/clip-vit-base-patch32"
bangla_text_model = "csebuetnlp/banglabert"  # Bangla text model
clip_model = CLIPModel.from_pretrained(clip_model_name)
processor = CLIPProcessor.from_pretrained(clip_model_name)
tokenizer = AutoTokenizer.from_pretrained(bangla_text_model)
# Load a pretrained Stable Diffusion model
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)  # Faster inference
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
# Preprocessing: Bangla Text to Prompt (Stable Diffusion Directly Supports Text Prompts)
def generate_image_from_text(bangla_text):
    """
    Generate an image from Bangla text using Stable Diffusion.
    """
    # Tokenize Bangla text (Optional: Use CLIP for embeddings if required)
    inputs = tokenizer(bangla_text, return_tensors="pt", padding=True, truncation=True)
    # Ensure tensors are moved to the same device as the pipeline
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    # Generate image directly from Bangla text
    image = pipe(prompt=bangla_text).images[0]
    return image
# Example Usage
if name == "main":
    bangla_text = "একটি সুন্দর প্রাকৃতিক দৃশ্য"  # A beautiful natural scene
    image = generate_image_from_text(bangla_text)
    # Save the generated image
    image.save("generated_image.png")
    print("Image has been saved as 'generated_image.png'.")

In [4]:
import torch
from transformers import CLIPProcessor, CLIPModel, AutoTokenizer, AutoModel
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from scipy.spatial.distance import cosine
import numpy as np

class BanglaBertImageGenerator:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Initialize BanglaBERT
        self.bangla_model = AutoModel.from_pretrained("csebuetnlp/banglabert")
        self.bangla_tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")
        self.bangla_model = self.bangla_model.to(self.device)

        # Initialize CLIP
        self.clip_model_name = "openai/clip-vit-base-patch32"
        self.clip_model = CLIPModel.from_pretrained(self.clip_model_name)
        self.clip_processor = CLIPProcessor.from_pretrained(self.clip_model_name)
        self.clip_model = self.clip_model.to(self.device)

        # Initialize Stable Diffusion
        self.pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            self.pipe.scheduler.config,
            use_karras_sigmas=True
        )
        self.pipe = self.pipe.to(self.device)

        # Define common translation mappings
        self.translation_map = {
            'প্রাকৃতিক': 'natural',
            'সুন্দর': 'beautiful',
            'দৃশ্য': 'scene',
            'পাহাড়': 'mountain',
            'সমুদ্র': 'ocean',
            'আকাশ': 'sky',
            # Add more mappings as needed
        }

    def _get_banglabert_embedding(self, text):
        """Get contextual embeddings from BanglaBERT"""
        inputs = self.bangla_tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(self.device)

        with torch.no_grad():
            outputs = self.bangla_model(**inputs)
            # Use [CLS] token embedding as text representation
            embedding = outputs.last_hidden_state[:, 0, :]

        return embedding

    def _enhance_prompt(self, bangla_text, embedding):
        """Create an enhanced prompt using BanglaBERT understanding"""
        # Basic word-by-word translation
        words = bangla_text.split()
        translated_words = []

        for word in words:
            if word in self.translation_map:
                translated_words.append(self.translation_map[word])
            else:
                # Keep original word if no translation available
                translated_words.append(word)

        base_translation = " ".join(translated_words)

        # Add quality enhancements based on content
        style_keywords = []
        if any(word in bangla_text for word in ['প্রাকৃতিক', 'দৃশ্য']):  # nature-related
            style_keywords.extend([
                "high resolution photography",
                "dramatic lighting",
                "professional landscape",
                "8k UHD",
                "detailed"
            ])

        # Combine everything into an enhanced prompt
        enhanced_prompt = f"{base_translation}, {', '.join(style_keywords)}"

        return enhanced_prompt

    def generate_image(self, bangla_text, num_images=1, num_inference_steps=50, guidance_scale=7.5):
        """Generate image(s) from Bangla text with enhanced understanding"""
        try:
            # Get BanglaBERT embedding
            embedding = self._get_banglabert_embedding(bangla_text)

            # Create enhanced prompt
            enhanced_prompt = self._enhance_prompt(bangla_text, embedding)

            # Set up negative prompt for better quality
            negative_prompt = "blurry, bad quality, distorted, pixelated, low resolution, oversaturated, undersaturated"

            # Generate image with enhanced settings
            with torch.autocast(self.device.type):
                result = self.pipe(
                    prompt=enhanced_prompt,
                    negative_prompt=negative_prompt,
                    num_images_per_prompt=num_images,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale
                )

            return result.images, enhanced_prompt

        except Exception as e:
            print(f"Error during image generation: {str(e)}")
            return None, None

# Example usage
if __name__ == "__main__":
    # Initialize generator
    generator = BanglaBertImageGenerator()

    # Example Bangla text
    bangla_text = "একটি সুন্দর প্রাকৃতিক দৃশ্য"  # A beautiful natural scene

    # Generate image with enhanced settings
    images, used_prompt = generator.generate_image(
        bangla_text,
        num_images=1,
        num_inference_steps=50,
        guidance_scale=8.0
    )

    # Save generated images
    if images:
        for idx, image in enumerate(images):
            image.save(f"generated_image_{idx}.png")
            print(f"Image {idx} saved with prompt: {used_prompt}")
    else:
        print("Image generation failed")

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Image 0 saved with prompt: একটি beautiful natural scene, high resolution photography, dramatic lighting, professional landscape, 8k UHD, detailed


#Result with open ai clip

In [8]:
import torch
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import numpy as np
from typing import List, Tuple, Optional

class EnhancedClipSDGenerator:
    def __init__(self, device: Optional[torch.device] = None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Initialize CLIP model and processor
        self.clip_model_name = "openai/clip-vit-base-patch32"
        self.clip_model = CLIPModel.from_pretrained(self.clip_model_name)
        self.processor = CLIPProcessor.from_pretrained(self.clip_model_name)
        self.tokenizer = CLIPTokenizer.from_pretrained(self.clip_model_name)

        # Enhanced Stable Diffusion initialization
        self.pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            safety_checker=None
        )
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            self.pipe.scheduler.config,
            use_karras_sigmas=True,
            algorithm_type="dpmsolver++"
        )
        self.pipe = self.pipe.to(self.device)
        self.clip_model = self.clip_model.to(self.device)

        # Initialize translation mapping
        self.translation_map = {
            'প্রাকৃতিক': 'natural',
            'সুন্দর': 'beautiful',
            'দৃশ্য': 'scene',
            'পাহাড়': 'mountain',
            'সূর্য': 'sun',
            'আকাশ': 'sky',
            'মেঘ': 'cloud',
            'নদী': 'river',
            'সমুদ্র': 'ocean',
            'গাছ': 'tree',
            'ফুল': 'flower',
            'মানুষ': 'person',
            'শহর': 'city',
            'গ্রাম': 'village',
            'বাড়ি': 'house'
        }

    def _get_text_embedding(self, text: str) -> torch.Tensor:
        """Get text embeddings using CLIP"""
        inputs = self.processor(
            text=text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            text_features = self.clip_model.get_text_features(**inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        return text_features

    def _translate_and_enhance_prompt(self, original_text: str) -> str:
        """Enhanced prompt generation with contextual understanding"""
        # Basic translation
        words = original_text.split()
        translated_parts = []
        for word in words:
            if word in self.translation_map:
                translated_parts.append(self.translation_map[word])
            else:
                translated_parts.append(word)

        base_translation = " ".join(translated_parts)

        # Base quality keywords
        style_keywords = [
            "high quality",
            "detailed",
            "professional photography",
            "4k",
            "sharp focus",
            "masterpiece"
        ]

        # Context-specific enhancements
        nature_keywords = ['প্রাকৃতিক', 'দৃশ্য', 'পাহাড়', 'নদী', 'সমুদ্র', 'গাছ']
        urban_keywords = ['শহর', 'বাড়ি', 'রাস্তা']
        sky_keywords = ['সূর্য', 'আকাশ', 'মেঘ']

        if any(word in original_text for word in nature_keywords):
            style_keywords.extend([
                "landscape photography",
                "dramatic lighting",
                "golden hour",
                "cinematic",
                "nature photography",
                "high resolution"
            ])

        if any(word in original_text for word in urban_keywords):
            style_keywords.extend([
                "urban photography",
                "architectural photography",
                "street photography",
                "modern"
            ])

        if any(word in original_text for word in sky_keywords):
            style_keywords.extend([
                "atmospheric",
                "beautiful sky",
                "natural lighting",
                "HDR",
                "vivid colors"
            ])

        # Combine into enhanced prompt
        enhanced_prompt = f"{base_translation}, {', '.join(style_keywords)}"
        return enhanced_prompt

    def generate_image(
        self,
        text: str,
        num_images: int = 1,
        num_inference_steps: int = 50,
        guidance_scale: float = 8.5,
        seed: Optional[int] = None
    ) -> Tuple[List[any], str]:
        """Generate images with enhanced settings"""
        try:
            if seed is not None:
                torch.manual_seed(seed)

            # Generate enhanced prompt
            enhanced_prompt = self._translate_and_enhance_prompt(text)

            # Enhanced negative prompt
            negative_prompt = (
                "blurry, low quality, pixelated, cartoon, fuzzy, noisy, "
                "oversaturated, undersaturated, distorted, deformed, "
                "bad anatomy, watermark, signature, text, low resolution, "
                "ugly, duplicate, morbid, mutilated, extra fingers, poorly drawn face, "
                "poorly drawn hands, missing fingers"
            )

            # Generate image
            with torch.autocast(self.device.type):
                result = self.pipe(
                    prompt=enhanced_prompt,
                    negative_prompt=negative_prompt,
                    num_images_per_prompt=num_images,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale
                )

            return result.images, enhanced_prompt

        except Exception as e:
            print(f"Error during image generation: {str(e)}")
            return None, None

# Example usage
if __name__ == "__main__":
    # Initialize generator
    generator = EnhancedClipSDGenerator()

    # Example texts
    texts = [
        "একটি সুন্দর প্রাকৃতিক দৃশ্য",  # A beautiful natural scene
        "সূর্যাস্তের সময় সমুদ্র তীরের দৃশ্য",  # Beach scene at sunset
        "পাহাড়ের চূড়ায় বরফ ঢাকা প্রাকৃতিক দৃশ্য"  # Snow-covered mountain peak
    ]

    # Generate images for each text
    for idx, text in enumerate(texts):
        images, used_prompt = generator.generate_image(
            text,
            num_images=1,
            num_inference_steps=50,
            guidance_scale=8.5,
            seed=42 + idx
        )

        if images:
            for img_idx, image in enumerate(images):
                image.save(f"generated_image_{idx}_{img_idx}.png")
                print(f"Image {idx}_{img_idx} saved with prompt: {used_prompt}")
        else:
            print(f"Image generation failed for text: {text}")

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  0%|          | 0/50 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (89 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['atmospheric , beautiful sky , natural lighting , hdr , vivid colors']


Image 0_0 saved with prompt: একটি beautiful natural scene, high quality, detailed, professional photography, 4k, sharp focus, masterpiece, landscape photography, dramatic lighting, golden hour, cinematic, nature photography, high resolution


  0%|          | 0/50 [00:00<?, ?it/s]

Image 1_0 saved with prompt: সূর্যাস্তের সময় ocean তীরের scene, high quality, detailed, professional photography, 4k, sharp focus, masterpiece, landscape photography, dramatic lighting, golden hour, cinematic, nature photography, high resolution, atmospheric, beautiful sky, natural lighting, HDR, vivid colors


  0%|          | 0/50 [00:00<?, ?it/s]

Image 2_0 saved with prompt: পাহাড়ের চূড়ায় বরফ ঢাকা natural scene, high quality, detailed, professional photography, 4k, sharp focus, masterpiece, landscape photography, dramatic lighting, golden hour, cinematic, nature photography, high resolution


#result with openai clip

In [9]:
import torch
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import numpy as np
from typing import List, Tuple, Optional

class BanglaSDGenerator:
    def __init__(self, device: Optional[torch.device] = None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Initialize CLIP model and processor
        self.clip_model_name = "openai/clip-vit-base-patch32"
        self.clip_model = CLIPModel.from_pretrained(self.clip_model_name)
        self.processor = CLIPProcessor.from_pretrained(self.clip_model_name)
        self.tokenizer = CLIPTokenizer.from_pretrained(self.clip_model_name)

        # Enhanced Stable Diffusion initialization
        self.pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            safety_checker=None
        )
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            self.pipe.scheduler.config,
            use_karras_sigmas=True,
            algorithm_type="dpmsolver++"
        )
        self.pipe = self.pipe.to(self.device)
        self.clip_model = self.clip_model.to(self.device)

    def _enhance_prompt(self, bangla_text: str) -> str:
        """Enhanced prompt generation with quality improvements for Bangla text"""
        # Quality keywords in Bangla
        quality_keywords = [
            "উচ্চ মানের",
            "বিস্তারিত",
            "পেশাদার ফটোগ্রাফি",
            "4k",
            "স্পষ্ট ফোকাস",
            "মাস্টারপিস",
            "সিনেমাটিক",
            "উচ্চ রেজোলিউশন",
            "নাটকীয় আলোকসজ্জা"
        ]

        # Combine into enhanced prompt
        enhanced_prompt = f"{bangla_text}, {', '.join(quality_keywords)}"
        return enhanced_prompt

    def generate_image(
        self,
        bangla_text: str,
        num_images: int = 1,
        num_inference_steps: int = 50,
        guidance_scale: float = 8.5,
        seed: Optional[int] = None
    ) -> Tuple[List[any], str]:
        """Generate images with Bangla prompt"""
        try:
            if seed is not None:
                torch.manual_seed(seed)

            # Generate enhanced prompt with Bangla
            enhanced_prompt = self._enhance_prompt(bangla_text)

            # Enhanced negative prompt (keeping in English for better results)
            negative_prompt = (
                "blurry, low quality, pixelated, cartoon, fuzzy, noisy, "
                "oversaturated, undersaturated, distorted, deformed, "
                "bad anatomy, watermark, signature, text, low resolution, "
                "ugly, duplicate, morbid, mutilated, extra fingers, poorly drawn face, "
                "poorly drawn hands, missing fingers"
            )

            # Generate image
            with torch.autocast(self.device.type):
                result = self.pipe(
                    prompt=enhanced_prompt,
                    negative_prompt=negative_prompt,
                    num_images_per_prompt=num_images,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale
                )

            return result.images, enhanced_prompt

        except Exception as e:
            print(f"Error during image generation: {str(e)}")
            return None, None

# Example usage
if __name__ == "__main__":
    # Initialize generator
    generator = BanglaSDGenerator()

    # Example text in Bangla
    text = "পাহাড় এবং সূর্যাস্তের একটি সুন্দর প্রাকৃতিক দৃশ্য"

    # Generate images
    images, used_prompt = generator.generate_image(
        text,
        num_images=2,
        num_inference_steps=50,
        guidance_scale=8.5,
        seed=42
    )

    # Save generated images
    if images:
        for idx, image in enumerate(images):
            image.save(f"generated_image_{idx}.png")
            print(f"Image {idx} saved with prompt: {used_prompt}")
    else:
        print("Image generation failed")

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
Token indices sequence length is longer than the specified maximum sequence length for this model (272 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['� ৃ শ ্ য , উচ ্ চ ম া ন ে র , ব ি স ্ ত া র ি ত , প ে শ া দ া

  0%|          | 0/50 [00:00<?, ?it/s]

Image 0 saved with prompt: পাহাড় এবং সূর্যাস্তের একটি সুন্দর প্রাকৃতিক দৃশ্য, উচ্চ মানের, বিস্তারিত, পেশাদার ফটোগ্রাফি, 4k, স্পষ্ট ফোকাস, মাস্টারপিস, সিনেমাটিক, উচ্চ রেজোলিউশন, নাটকীয় আলোকসজ্জা
Image 1 saved with prompt: পাহাড় এবং সূর্যাস্তের একটি সুন্দর প্রাকৃতিক দৃশ্য, উচ্চ মানের, বিস্তারিত, পেশাদার ফটোগ্রাফি, 4k, স্পষ্ট ফোকাস, মাস্টারপিস, সিনেমাটিক, উচ্চ রেজোলিউশন, নাটকীয় আলোকসজ্জা
